# Importing files and other python libraries

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import sqlite3

data_dir = Path.cwd()

df = pd.read_csv(data_dir / '32100053-eng' / '32100053.csv')

df.head(10)

C:\Users\Dhanan Thanos\AppData\Local\Temp\ipykernel_26576\923811285.py:10: DtypeWarning: Columns (0: SYMBOL) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_dir / '32100053-eng' / '32100053.csv')


,REF_DATE,GEO,DGUID,Commodity,Supply and disposition,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1960,Canada,2021A000011124,Wheat flour,Total supply,Tonnes,288,thousands,3,v108200,1.1.2,1934.34,NaN,NaN,NaN,2
1,1960,Canada,2021A000011124,Wheat flour,Beginning stocks,Tonnes,288,thousands,3,v108201,1.1.3,87.07,NaN,NaN,NaN,2
2,1960,Canada,2021A000011124,Wheat flour,Production,Tonnes,288,thousands,3,v108202,1.1.4,1847.19,NaN,NaN,NaN,2
3,1960,Canada,2021A000011124,Wheat flour,Imports,Tonnes,288,thousands,3,v108203,1.1.5,0.08,NaN,NaN,NaN,2
4,1960,Canada,2021A000011124,Wheat flour,Total disposition,Tonnes,288,thousands,3,v54326402,1.1.11,1842.27,NaN,NaN,NaN,2
5,1960,Canada,2021A000011124,Wheat flour,Domestic disappearance,Tonnes,288,thousands,3,v108199,1.1.1,1060.01,NaN,NaN,NaN,2
6,1960,Canada,2021A000011124,Wheat flour,Exports,Tonnes,288,thousands,3,v108205,1.1.7,715.56,NaN,NaN,NaN,2
7,1960,Canada,2021A000011124,Wheat flour,Manufacturing,Tonnes,288,thousands,3,v108206,1.1.8,45.07,NaN,NaN,NaN,2
8,1960,Canada,2021A000011124,Wheat flour,Waste,Tonnes,288,thousands,3,v108207,1.1.9,21.63,NaN,NaN,NaN,2
9,1960,Canada,2021A000011124,Wheat flour,Ending stocks,Tonnes,288,thousands,3,v108208,1.1.10,92.07,NaN,NaN,NaN,2


**Seems like there is no issues with importing the dataset. Onto the next part.**

# Checking for duplicates

In [3]:
df[df.duplicated()]

,REF_DATE,GEO,DGUID,Commodity,Supply and disposition,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS


Good, no duplicates to worry about.

# What do all the columns mean?
| Column | Description | Data Type |
|---|---|---|
| REF_DATE | Reference period for the series being released | Date(Year) |
| GEO | Geographic region | String |
| DGUID | Geography unique identifier | String |
| Commodity | Name of the item | String |
| Supply and Disposition | What part of the production line | String |
| UOM | Acronym for 'Unit of Measure' | String |
| UOM_ID | Numeric identifier for UOM | Int |
| SCALAR_FACTOR | Scaling factor (ex. thousands = X1000) | String |
| SCALAR_ID | Numeric reference for SCALAR_FACTOR (ex. 2 = 10^2 = 100) | Int |
| VECTOR | Time series identifier (Likely won't need this) | String |
| COORDINATE | Encoded combination of dimension members | String |
| VALUE | Numeric value recorded | Float |
| STATUS | Shows various states of a data value using symbol (Also likely not needed.) | Float |
| SYMBOL | Basically a legend. Likely won't need this.| Float |
| TERMINATED | Distinguishes data points that are no longer being updated. | Float |
| DECIMALS | How many decimal points in a given value | Int |


## There are a few columns that are pointless to the data analysis.
My propositions. Make a new data frame with less columns by:
1.  Removing SCALAR_FACTOR and SCALAR_ID after performing some multiplication on VALUE by SCALAR_FACTOR or SCALAR_ID.
2.  Remove GEO, this is a national dataset, it is an aggregate of all of the provinces.
3.  COORDINATE is useless for the purpose of this analysis. So it will be removed.
4.  STATUS and SYMBOL are pointless for this analysis.
5.  DECIMALS is mostly pointless.

In [12]:
df_trim = df[['REF_DATE', 'GEO', 'Commodity', 'Supply and disposition', 'UOM', 'SCALAR_ID', 'VALUE', 'TERMINATED']]

df_trim['VALUE'] = df_trim['VALUE'] * (10 ** df_trim['SCALAR_ID'])
df_trim = df_trim.drop(columns=['SCALAR_ID'])

df_trim.head()

,REF_DATE,GEO,Commodity,Supply and disposition,UOM,VALUE,TERMINATED
0,1960,Canada,Wheat flour,Total supply,Tonnes,1934340.0,NaN
1,1960,Canada,Wheat flour,Beginning stocks,Tonnes,87070.0,NaN
2,1960,Canada,Wheat flour,Production,Tonnes,1847190.0,NaN
3,1960,Canada,Wheat flour,Imports,Tonnes,80.0,NaN
4,1960,Canada,Wheat flour,Total disposition,Tonnes,1842270.0,NaN


## Let's do a quick preview of the data to see if anything stands out.

In [13]:
df_trim.info()

<class 'pandas.DataFrame'>
RangeIndex: 72543 entries, 0 to 72542
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   REF_DATE                72543 non-null  int64  
 1   GEO                     72543 non-null  str    
 2   Commodity               72543 non-null  str    
 3   Supply and disposition  72543 non-null  str    
 4   UOM                     72543 non-null  str    
 5   VALUE                   66289 non-null  float64
 6   TERMINATED              1227 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 3.9 MB


In [14]:
df_trim.describe(include= 'all')

,REF_DATE,GEO,Commodity,Supply and disposition,UOM,VALUE,TERMINATED
count,72543.000000,72543,72543,72543,72543,6.628900e+04,1227
unique,NaN,1,193,10,2,NaN,1
top,NaN,Canada,Wheat flour,Total disposition,Tonnes,NaN,t
freq,NaN,72543,660,12260,65841,NaN,1227
mean,1993.061095,NaN,NaN,NaN,NaN,1.220125e+05,NaN
std,18.660523,NaN,NaN,NaN,NaN,3.647269e+05,NaN
min,1960.000000,NaN,NaN,NaN,NaN,0.000000e+00,NaN
25%,1977.000000,NaN,NaN,NaN,NaN,3.630000e+03,NaN
50%,1993.000000,NaN,NaN,NaN,NaN,1.831000e+04,NaN
75%,2009.000000,NaN,NaN,NaN,NaN,7.277000e+04,NaN
